# ε-жадный алгоритм для стационарного многорукого бандита

У бандита k действий с фиксированными неизвестными распределениями награды. На каждом шаге выбираем действие и наблюдаем только его награду. Цель — максимизировать ожидаемую суммарную награду.

## Жадный алгоритм

Жадный выбор использует $A_t = \arg\max_a Q_t(a)$.

## ε-жадный алгоритм

С вероятностью ε выбираем случайное действие, иначе действуем жадно. После наблюдения номер $N_n$ оценка обновляется средним: $Q_{n+1}=Q_n+(R_n-Q_n)/N_n$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from rl_fun.algorithms.bandits.epsilon_greedy import run_epsilon_greedy
from rl_fun.algorithms.bandits.greedy import run_greedy
from rl_fun.algorithms.bandits.random_agent import run_random
from rl_fun.environments.bandit import StationaryBanditEnv
from rl_fun.experiments.compare import aggregate_batch
from rl_fun.experiments.config import (
    AlgorithmConfig,
    EnvironmentConfig,
    ExperimentConfig,
    RunConfig,
)
from rl_fun.experiments.runner import run_many
from rl_fun.tracking.metrics import MemoryMetricSink

demo_env = StationaryBanditEnv(arms=5, horizon=20)
observation, info = demo_env.reset(seed=7)
observation, reward, terminated, truncated, info = demo_env.step(0)
demo_env.close()
reward, info

In [ ]:
def run_for_notebook(name, seed=42, steps=500):
    env = StationaryBanditEnv(arms=10, horizon=steps)
    env.reset(seed=seed)
    sink = MemoryMetricSink()
    rng = np.random.default_rng(seed)
    if name == "random":
        outcome = run_random(env, steps, rng, sink)
    elif name == "greedy":
        outcome = run_greedy(env, steps, rng, sink)
    else:
        outcome = run_epsilon_greedy(env, steps, rng, sink, epsilon=0.1)
    env.close()
    return outcome, sink


results = {name: run_for_notebook(name) for name in ("random", "greedy", "epsilon_greedy")}

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = {
    "random": "Случайный выбор",
    "greedy": "Жадный алгоритм",
    "epsilon_greedy": "ε-жадный алгоритм",
}
for name, (_, sink) in results.items():
    steps = [event.step for event in sink.events]
    axes[0].plot(
        steps,
        [event.metrics["train/cumulative_reward"] for event in sink.events],
        label=labels[name],
    )
    axes[1].plot(
        steps,
        [event.metrics["train/cumulative_regret"] for event in sink.events],
        label=labels[name],
    )
axes[0].set_title("Суммарная награда")
axes[0].set_ylabel("Награда")
axes[1].set_title("Накопленное сожаление")
axes[1].set_ylabel("Сожаление")
for axis in axes:
    axis.set_xlabel("Шаг")
    axis.legend()
plt.show()

Один seed иллюстрирует поведение, но не доказывает превосходство алгоритма. Случайные награды могут изменить видимый результат: сравнивайте несколько независимо инициализированных запусков.

In [ ]:
batch_config = ExperimentConfig(
    name="notebook-epsilon-greedy",
    environment=EnvironmentConfig(
        id="RLFun/StationaryBandit-v0",
        kwargs={"arms": 10, "reward_std": 1.0},
    ),
    algorithm=AlgorithmConfig(id="bandit_epsilon", kwargs={"epsilon": 0.1}),
    run=RunConfig(seeds=tuple(range(10)), total_steps=500, workers=1, output_root="../../runs"),
)
aggregate_batch(run_many(batch_config), "train/cumulative_reward")

## Что попробовать дальше

1. Измените ε и сравните исследование действий с использованием текущих оценок.
2. Измените шум награды и неопределённость оценок.
3. Измените число действий и сложность исследования.